# 03 — PyTorch Experiments: MLP & LSTM Price Forecasting

Explores neural network alternatives to XGBoost for price prediction.
Does **not** affect the production pipeline — use findings here to decide
whether a neural approach is worth replacing XGBoost in `05_train_price_model.py`.

**Models explored:**
- `MLP` — simple feedforward network on the 13 tabular features
- `PriceLSTM` — sequence model consuming a 7-day price window per ASIN

**Tracked with local MLflow** — `uv run mlflow ui` from `local/`.

In [ ]:
import sys
import os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
SRC_ROOT  = os.path.join(REPO_ROOT, "databricks")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch
from torch.utils.data import DataLoader, TensorDataset

from src.evaluation.metrics import compute_metrics
from src.features.definitions import ALL_FEATURE_COLS, TARGET_COL
from src.models.hyperparams import DEFAULT_PARAMS
from src.models.price_optimizer import PriceOptimizer

mlflow.set_tracking_uri(os.path.join(os.getcwd(), "../mlruns"))
mlflow.set_experiment("pricesense-pytorch-experiments")

DATA_DIR   = os.path.join(os.getcwd(), "../data")
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

## Load and prepare tabular features

In [ ]:
raw = pd.read_csv(os.path.join(DATA_DIR, "sample_gold_features.csv"), parse_dates=["scrape_date"])
raw = raw.dropna(subset=[TARGET_COL, "current_price", "comp_median_price"])

df = raw.sort_values(["asin", "scrape_date"]).copy()
df["day_of_week"] = df["scrape_date"].dt.dayofweek
df["price_to_median_ratio"] = df["current_price"] / df["comp_median_price"].replace(0, np.nan)

for col in ALL_FEATURE_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

df_sorted = df.sort_values("scrape_date")
TEST_SIZE = 0.2
split_idx = int(len(df_sorted) * (1 - TEST_SIZE))

X = df_sorted[ALL_FEATURE_COLS].astype(float).values
y = df_sorted[TARGET_COL].astype(float).values

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Normalise features for neural networks
X_mean, X_std = X_train.mean(axis=0), X_train.std(axis=0) + 1e-8
X_train_norm = (X_train - X_mean) / X_std
X_test_norm  = (X_test  - X_mean) / X_std

print(f"Train: {X_train.shape}  Test: {X_test.shape}")

## Model 1: MLP on tabular features

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, hidden: int = 64, dropout: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


def train_mlp(X_tr, y_tr, X_te, y_te, epochs=100, lr=1e-3, hidden=64, batch_size=32):
    dataset = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.float32),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model  = MLP(X_tr.shape[1], hidden=hidden).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    train_losses, test_rmses = [], []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(loader))

        model.eval()
        with torch.no_grad():
            X_te_t = torch.tensor(X_te, dtype=torch.float32).to(DEVICE)
            preds = model(X_te_t).cpu().numpy()
        test_rmses.append(compute_metrics(y_te, preds)["rmse"])

    return model, train_losses, test_rmses


with mlflow.start_run(run_name="MLP-tabular"):
    mlp, train_losses, test_rmses = train_mlp(
        X_train_norm, y_train, X_test_norm, y_test, epochs=100, lr=1e-3, hidden=64
    )
    mlp.eval()
    with torch.no_grad():
        mlp_preds = mlp(torch.tensor(X_test_norm, dtype=torch.float32).to(DEVICE)).cpu().numpy()

    mlp_metrics = compute_metrics(y_test, mlp_preds)
    mlflow.log_metrics(mlp_metrics)
    print(f"MLP — RMSE: {mlp_metrics['rmse']:.4f}  MAPE: {mlp_metrics['mape']:.2f}%")

## MLP training curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses)
axes[0].set_title("MLP Training Loss (MSE)")
axes[0].set_xlabel("Epoch")
axes[1].plot(test_rmses)
axes[1].set_title("MLP Test RMSE per Epoch")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

## Model 2: LSTM on 7-day price sequences

Frame the problem as a sequence: given the last 7 days of `[price, comp_median, comp_pressure]`
for an ASIN, predict tomorrow's `suggested_price`.

In [ ]:
SEQ_LEN     = 7
SEQ_FEATURES = ["current_price", "comp_median_price", "comp_pressure_score"]

def build_sequences(df_in, seq_len=7):
    X_seq, y_seq = [], []
    for _, grp in df_in.groupby("asin"):
        grp = grp.sort_values("scrape_date").reset_index(drop=True)
        for i in range(seq_len, len(grp)):
            window = grp[SEQ_FEATURES].iloc[i - seq_len:i].values
            target = grp[TARGET_COL].iloc[i]
            if not np.isnan(window).any() and not np.isnan(target):
                X_seq.append(window)
                y_seq.append(target)
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

X_seq, y_seq = build_sequences(df, SEQ_LEN)
split_seq    = int(len(X_seq) * (1 - TEST_SIZE))
X_seq_tr, X_seq_te = X_seq[:split_seq], X_seq[split_seq:]
y_seq_tr, y_seq_te = y_seq[:split_seq], y_seq[split_seq:]

# Normalise per feature across time
seq_mean = X_seq_tr.mean(axis=(0, 1))
seq_std  = X_seq_tr.std(axis=(0, 1)) + 1e-8
X_seq_tr = (X_seq_tr - seq_mean) / seq_std
X_seq_te = (X_seq_te - seq_mean) / seq_std

print(f"Sequence train: {X_seq_tr.shape}  test: {X_seq_te.shape}")

In [ ]:
class PriceLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int = 32, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head  = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def train_lstm(X_tr, y_tr, X_te, y_te, epochs=100, lr=5e-4):
    dataset = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    loader  = DataLoader(dataset, batch_size=16, shuffle=True)

    model   = PriceLSTM(input_size=X_tr.shape[2]).to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    test_rmses = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            preds = model(torch.from_numpy(X_te).to(DEVICE)).cpu().numpy()
        test_rmses.append(compute_metrics(y_te, preds)["rmse"])

    return model, test_rmses


with mlflow.start_run(run_name="LSTM-sequence"):
    lstm, lstm_rmses = train_lstm(X_seq_tr, y_seq_tr, X_seq_te, y_seq_te, epochs=100)
    lstm.eval()
    with torch.no_grad():
        lstm_preds = lstm(torch.from_numpy(X_seq_te).to(DEVICE)).cpu().numpy()

    lstm_metrics = compute_metrics(y_seq_te, lstm_preds)
    mlflow.log_metrics(lstm_metrics)
    print(f"LSTM — RMSE: {lstm_metrics['rmse']:.4f}  MAPE: {lstm_metrics['mape']:.2f}%")

## Final comparison: XGBoost vs MLP vs LSTM

In [ ]:
xgb_opt   = PriceOptimizer(DEFAULT_PARAMS).fit(
    pd.DataFrame(X_train, columns=ALL_FEATURE_COLS),
    pd.Series(y_train),
)
xgb_preds = xgb_opt.predict(pd.DataFrame(X_test, columns=ALL_FEATURE_COLS))
xgb_metrics = compute_metrics(y_test, xgb_preds)

summary = pd.DataFrame([
    {"model": "XGBoost (tabular)", **xgb_metrics},
    {"model": "MLP (tabular)",     **mlp_metrics},
    {"model": "LSTM (sequence)",   **lstm_metrics},
]).sort_values("rmse")

print("=== Final comparison ===")
print(summary.round(4).to_string(index=False))
print()
winner = summary.iloc[0]["model"]
print(f"Best model: {winner}")
print()
print("Decision guide:")
print("  XGBoost wins → keep production as-is (fast, interpretable)")
print("  MLP wins     → consider replacing XGBoost; add to 05_train_price_model.py")
print("  LSTM wins    → price has strong temporal structure; consider a sequence model")

## LSTM convergence

In [ ]:
plt.plot(lstm_rmses)
plt.title("LSTM Test RMSE per Epoch")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.tight_layout()
plt.show()